In [2]:
import cv2
import numpy as np
import glob
import os
ESCALA_PROC = 1200
CLAHE_CLIP = 4.0
CLAHE_TILE = 4
UMBRAL = 120
AREA_MIN = 40
AREA_MAX = 500
ASP_MIN = 0.65
ASP_MAX = 1.54
DIM_MIN = 5
DIM_MAX = 25
BLOB_AREA_MIN = 200
BLOB_AREA_MAX = 5000
EPS = 33
ROI_FACTOR = 3.0
OUTLIER_RATIO = 1.2
BORDE_SUP = 0.15
BORDE_INF = 0.15
BORDE_LAT = 0.03
MARGEN_BOX = 35
def dbscan_numpy(pts, eps):
    n = len(pts)
    labels = np.full(n, -1, dtype=int)
    visited = np.zeros(n, dtype=bool)
    cluster = 0
    diff = pts[:, None, :] - pts[None, :, :]
    dist = np.sqrt((diff ** 2).sum(axis=2))
    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        nb = np.where(dist[i] <= eps)[0].tolist()
        labels[i] = cluster
        seeds = [v for v in nb if v != i]
        while seeds:
            j = seeds.pop()
            if not visited[j]:
                visited[j] = True
                nb2 = np.where(dist[j] <= eps)[0].tolist()
                for k in nb2:
                    if k not in seeds:
                        seeds.append(k)
            if labels[j] == -1:
                labels[j] = cluster
        cluster += 1
    return labels
def contar_puntos_roi(gray_eq, cx, cy, radio):
    x1 = max(0, int(cx - radio))
    y1 = max(0, int(cy - radio))
    x2 = min(gray_eq.shape[1], int(cx + radio))
    y2 = min(gray_eq.shape[0], int(cy + radio))
    roi = gray_eq[y1:y2, x1:x2]
    if roi.size == 0:
        return 0
    conteos = {}
    k2 = np.ones((2, 2), np.uint8)
    for t in range(75, 140, 5):
        _, m = cv2.threshold(roi, t, 255, cv2.THRESH_BINARY_INV)
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k2)
        lbl = cv2.connectedComponents(m, 8)[1]
        borde = set()
        borde.update(lbl[0, :].tolist())
        borde.update(lbl[-1, :].tolist())
        borde.update(lbl[:, 0].tolist())
        borde.update(lbl[:, -1].tolist())
        n_comp, _, se, _ = cv2.connectedComponentsWithStats(m, 8)
        validos = [
            j for j in range(1, n_comp)
            if j not in borde
            and 10 < se[j, cv2.CC_STAT_AREA] < 400
            and 4 <= se[j, cv2.CC_STAT_WIDTH] <= 28
            and 4 <= se[j, cv2.CC_STAT_HEIGHT] <= 28
            and 0.35 < se[j, cv2.CC_STAT_WIDTH] / max(se[j, cv2.CC_STAT_HEIGHT], 1) < 2.8
        ]
        n = len(validos)
        if 1 <= n <= 6:
            conteos[n] = conteos.get(n, 0) + 1
    return max(conteos, key=conteos.get) if conteos else 0
def procesar_imagen(ruta_imagen, guardar=False, carpeta_salida="resultados_blancos", mostrar=False):
    arr = np.fromfile(ruta_imagen, np.uint8)
    img_orig = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img_orig is None:
        print(f"[ERROR] No se pudo leer: {ruta_imagen}")
        return 0, []
    h, w = img_orig.shape[:2]
    escala = ESCALA_PROC / max(h, w)
    img = cv2.resize(img_orig, (int(w * escala), int(h * escala)))
    ih, iw = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(CLAHE_TILE, CLAHE_TILE))
    gray_eq = clahe.apply(gray)
    _, mask = cv2.threshold(gray_eq, UMBRAL, 255, cv2.THRESH_BINARY_INV)
    sup = int(ih * BORDE_SUP)
    inf = int(ih * (1.0 - BORDE_INF))
    lat = int(iw * BORDE_LAT)
    mask[:sup, :] = 0
    mask[inf:, :] = 0
    mask[:, :lat] = 0
    mask[:, iw - lat:] = 0
    k2 = np.ones((2, 2), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k2)
    _, _, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    n_comp = stats.shape[0]
    puntos = []
    bws_map = {}
    blobs = []
    for i in range(1, n_comp):
        area = stats[i, cv2.CC_STAT_AREA]
        bw = stats[i, cv2.CC_STAT_WIDTH]
        bh = stats[i, cv2.CC_STAT_HEIGHT]
        cx, cy = centroids[i]
        asp = bw / max(bh, 1)
        if (AREA_MIN < area < AREA_MAX and ASP_MIN < asp < ASP_MAX and DIM_MIN <= bw <= DIM_MAX and DIM_MIN <= bh <= DIM_MAX):
            bws_map[len(puntos)] = bw
            puntos.append((cx, cy))
        elif (BLOB_AREA_MIN <= area < BLOB_AREA_MAX and 0.35 < asp < 2.8 and (bw > 25 or bh > 25) and sup < cy < inf):
            blobs.append((cx, cy, bw, bh))
    if not puntos:
        return 0, []
    pts_arr = np.array(puntos, dtype=np.float32)
    labels = dbscan_numpy(pts_arr, eps=EPS)
    dados = []
    dados_pos = []
    for c in set(labels):
        idxs = list(np.where(labels == c)[0])
        m = pts_arr[idxs]
        n_d = len(m)
        if n_d < 1:
            continue
        cx_c = float(m[:, 0].mean())
        cy_c = float(m[:, 1].mean())
        bws_c = np.array([bws_map.get(i, 9) for i in idxs], dtype=float)
        p75 = float(np.percentile(bws_c, 75))
        bws_l = bws_c[bws_c <= p75 * 1.2]
        med_bw = float(np.median(bws_l)) if len(bws_l) > 0 else float(np.median(bws_c))
        radio = int(med_bw * ROI_FACTOR)
        radio = min(radio, 75)
        n_roi = contar_puntos_roi(gray_eq, cx_c, cy_c, radio)
        hay_outlier = (bws_c.max() > med_bw * OUTLIER_RATIO or bws_c.min() < med_bw / OUTLIER_RATIO)
        if hay_outlier and 1 <= n_roi <= 6:
            val = n_roi
        elif n_roi == 0 or n_roi == n_d + 1:
            val = n_d if 1 <= n_d <= 6 else 0
        elif n_roi >= n_d + 2:
            val = n_roi if n_roi <= 6 else n_d
        else:
            val = n_d if 1 <= n_d <= 6 else 0
        if 1 <= val <= 6:
            dados.append({'valor': val, 'cx': cx_c, 'cy': cy_c, 'pts': m})
            dados_pos.append((cx_c, cy_c))
    for (bcx, bcy, bw, bh) in blobs:
        ya = any(np.sqrt((bcx - dx) ** 2 + (bcy - dy) ** 2) < 70 for dx, dy in dados_pos)
        if not ya:
            radio = min(int(max(bw, bh) / 2 + 20), 75)
            n = contar_puntos_roi(gray_eq, bcx, bcy, radio)
            if 1 <= n <= 6:
                dados.append({'valor': n, 'cx': bcx, 'cy': bcy, 'pts': np.array([[bcx, bcy]], dtype=np.float32)})
                dados_pos.append((bcx, bcy))
    lista_valores = sorted(d['valor'] for d in dados)
    if guardar or mostrar:
        vis = img.copy()
        for d in dados:
            m = d['pts']
            x1 = max(0, int(m[:, 0].min()) - MARGEN_BOX)
            y1 = max(0, int(m[:, 1].min()) - MARGEN_BOX)
            x2 = min(iw, int(m[:, 0].max()) + MARGEN_BOX)
            y2 = min(ih, int(m[:, 1].max()) + MARGEN_BOX)
            cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 60, 0), 2)
            lbl = str(d['valor'])
            fs, th = 1.0, 2
            (tw, lh), base = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, fs, th)
            lx, ly = x1 + 4, y1 + lh + 4
            cv2.rectangle(vis, (lx - 2, ly - lh - 2), (lx + tw + 2, ly + base), (255, 255, 255), cv2.FILLED)
            cv2.putText(vis, lbl, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, fs, (0, 0, 200), th, cv2.LINE_AA)
        if guardar:
            os.makedirs(carpeta_salida, exist_ok=True)
            base = os.path.basename(ruta_imagen)
            salida = os.path.join(carpeta_salida, f"res_{base}")
            cv2.imwrite(salida, vis)
        if mostrar:
            cv2.imshow(f"Resultado: {os.path.basename(ruta_imagen)}", vis)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
    return len(dados), lista_valores
PATRON = "img_blancos/*.jpg"
GUARDAR = True
CARPETA_SALIDA = "resultados_blancos"
MOSTRAR = False
archivos = sorted(glob.glob(PATRON))
if not archivos:
    print(f"No se encontraron imágenes con el patrón: {PATRON}")
else:
    sep = "─" * 72
    print(sep)
    print(f"Procesando {len(archivos)} imagen(es) | Guardar: {GUARDAR} → '{CARPETA_SALIDA}'")
    print(sep)
    total_dados = 0
    suma_total = 0
    for archivo in archivos:
        nombre = os.path.basename(archivo)
        n, caras = procesar_imagen(archivo, guardar=GUARDAR, carpeta_salida=CARPETA_SALIDA, mostrar=MOSTRAR)
        total_dados += n
        suma_total += sum(caras)
        print(f"{nombre:<45s} dados:{n:2d} caras:{str(caras):<22s} suma:{sum(caras)}")
    print(sep)
    print(f"Total dados detectados : {total_dados}")
    print(f"Suma total de puntos : {suma_total}")
    print(sep)

────────────────────────────────────────────────────────────────────────
Procesando 7 imagen(es) | Guardar: True → 'resultados_blancos'
────────────────────────────────────────────────────────────────────────
IMG_20191209_100701.jpg                       dados: 4 caras:[2, 4, 5, 5]           suma:16
IMG_20191209_100706.jpg                       dados: 4 caras:[4, 5, 5, 6]           suma:20
IMG_20191209_100714.jpg                       dados: 5 caras:[2, 2, 3, 3, 3]        suma:13
IMG_20191209_100719.jpg                       dados: 5 caras:[1, 4, 4, 4, 6]        suma:19
IMG_20191209_100722.jpg                       dados: 5 caras:[3, 4, 4, 5, 6]        suma:22
IMG_20191209_100728.jpg                       dados: 5 caras:[2, 4, 4, 5, 6]        suma:21
IMG_20191209_100733.jpg                       dados: 5 caras:[3, 5, 6, 6, 6]        suma:26
────────────────────────────────────────────────────────────────────────
Total dados detectados : 33
Suma total de puntos : 137
───────────────────